# CineMatch: Hybrid Movie Recommendation Engine
**Built by Jatin | [ijatin.dev](https://ijatin.dev)**

This project explores recommendation systems through the lens of Human-Centred AI. It evaluates the trade-offs between traditional statistical methods (TF-IDF) and Deep Learning transformers (BERT) for content discovery.

### 🎯 Key Features
* **Item-Item Recommendation**: Finding visually and thematically similar movies.
* **Personalization Engine**: Generating dynamic user profiles based on weighted historical ratings.
* **Algorithmic Evaluation**: Measuring system performance using Precision@10.
* **Production-Ready Export**: Saving API artifacts and model statistics for a Flask frontend.

### 💡 Why This Matters
Modern platforms like Netflix, Spotify, and Amazon rely heavily on understanding not just *what* a user clicks, but *why*. 
* **Statistical Models (TF-IDF)** map explicit keywords (e.g., 'Space', 'Action'). They are incredibly fast and highly explainable, building user trust.
* **Semantic Models (BERT)** capture the hidden 'vibe' or implicit context of a description, finding matches even when exact keywords differ.

Understanding when to deploy each architecture is crucial for building scalable, engaging, and transparent recommendation systems.

In [46]:
# ================================
# 1. IMPORTS
# ================================
import pandas as pd
import numpy as np
import re
import ast
import os
import json

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ================================
# 2. LOAD DATA
# ================================
ratings = pd.read_csv("/kaggle/input/datasets/markshah/movielens/ratings.csv")
movies = pd.read_csv("/kaggle/input/datasets/markshah/movielens/movies.csv")
tags = pd.read_csv("/kaggle/input/datasets/markshah/movielens/tags.csv")
links = pd.read_csv("/kaggle/input/datasets/markshah/movielens/links.csv")
tmdb = pd.read_csv("/kaggle/input/datasets/markshah/tmdb-data/tmdb_5000_movies.csv")

## Dataset Overview
To generate meaningful recommendations, we need rich context. The standard MovieLens dataset provides excellent collaborative filtering signals (ratings) but lacks descriptive depth. By linking it with TMDB via `tmdbId`, we unlock detailed plot overviews, extensive genre tags, and thematic keywords.

## Feature Engineering
We merge and clean the data to create a unified `final_features` column. This column combines genres, user-generated tags, TMDB keywords, and plot overviews into a single string. This comprehensive "document" serves as the foundation for both our statistical and deep learning models.

In [47]:
# ================================
# 3. CLEAN TAGS
# ================================
tags['tag'] = tags['tag'].astype(str).str.lower().str.strip()

# ================================
# 4. AGGREGATE TAGS
# ================================
tags_grouped = tags.groupby('movieId')['tag'].apply(lambda x: " ".join(x)).reset_index()

# ================================
# 5. MERGE MOVIELENS
# ================================
movies_with_tags = movies.merge(tags_grouped, on='movieId', how='left')
movies_with_tags['tag'] = movies_with_tags['tag'].fillna("")
movies_with_tags['genres'] = movies_with_tags['genres'].str.replace("|", " ")

# ================================
# 6. CLEAN TEXT
# ================================
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

movies_with_tags['features'] = movies_with_tags['genres'] + " " + movies_with_tags['tag']
movies_with_tags['features_clean'] = movies_with_tags['features'].apply(clean_text)

# ================================
# 7. PREPARE TMDB LINKS
# ================================
links = links.dropna(subset=['tmdbId'])
links['tmdbId'] = links['tmdbId'].astype(int)

# ================================
# 8. MERGE WITH TMDB
# ================================
movies_with_tmdb = movies_with_tags.merge(
    links[['movieId', 'tmdbId']],
    on='movieId',
    how='inner'
)

movies_with_tmdb = movies_with_tmdb.merge(
    tmdb,
    left_on='tmdbId',
    right_on='id',
    how='inner'
)

# ================================
# 9. FIX COLUMN NAMES
# ================================
movies_with_tmdb = movies_with_tmdb.rename(columns={'title_x': 'title'})
movies_with_tmdb = movies_with_tmdb.drop(columns=['title_y'], errors='ignore')

# ================================
# 10. PARSE TMDB JSON
# ================================
def extract_names(text):
    try:
        items = ast.literal_eval(text)
        return " ".join([i['name'] for i in items])
    except:
        return ""

movies_with_tmdb['tmdb_keywords'] = movies_with_tmdb['keywords'].apply(extract_names)
movies_with_tmdb['tmdb_genres'] = movies_with_tmdb['genres_y'].apply(extract_names)

# ================================
# 11. CLEAN OVERVIEW
# ================================
movies_with_tmdb['overview'] = movies_with_tmdb['overview'].fillna("")
movies_with_tmdb['overview'] = movies_with_tmdb['overview'].apply(clean_text)

# ================================
# 12. FINAL FEATURES
# ================================
movies_with_tmdb['final_features'] = (
    movies_with_tmdb['features_clean'] + " " +
    movies_with_tmdb['overview'] + " " +
    movies_with_tmdb['tmdb_keywords'] + " " +
    movies_with_tmdb['tmdb_genres']
)

# ================================
# 13. RESET INDEX
# ================================
movies_with_tmdb = movies_with_tmdb.reset_index(drop=True)

## TF-IDF Vectorization
Term Frequency-Inverse Document Frequency (TF-IDF) transforms our text into a sparse mathematical matrix. It heavily weights unique, identifying words while penalizing common, uninformative ones. This approach is highly effective for explicit keyword matching.

## TF-IDF Similarity
With our sparse matrix prepared, we calculate the Cosine Similarity between vectors to determine how closely related two movies are based on their metadata. We also establish a baseline popularity metric to prevent obscure niche films from dominating the results.



In [48]:
# ================================
# 14. TF-IDF VECTORIZATION
# ================================
tfidf = TfidfVectorizer(
    max_features=15000,
    stop_words='english',
    ngram_range=(1,2)
)

tfidf_matrix = tfidf.fit_transform(movies_with_tmdb['final_features'])

# ================================
# 15. INDEX MAPPING
# ================================
indices = pd.Series(
    movies_with_tmdb.index,
    index=movies_with_tmdb['title']
).drop_duplicates()

# ================================
# 16. POPULARITY CALCULATION
# ================================
movie_popularity = ratings.groupby('movieId').size().reset_index(name='count')

movies_with_tmdb = movies_with_tmdb.merge(movie_popularity, on='movieId', how='left')
movies_with_tmdb['count'] = movies_with_tmdb['count'].fillna(1)


## TF-IDF Recommendations
Let's test our baseline statistical model by querying a well-known family film. The model relies strictly on intersecting keywords and tags to find comparable items.

In [49]:
# ================================
# 17. CONTENT-BASED RECOMMENDER
# ================================
def recommend(title, top_n=10):
    if title not in indices:
        return f"{title} not found"
    
    idx = indices[title]
    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    
    sim_indices = sim_scores.argsort()[-(top_n+1):][::-1][1:]
    
    return movies_with_tmdb['title'].iloc[sim_indices]

# Formatter function to clean up raw list outputs
def print_formatted_recs(title, recs):
    print(f"Top Recommendations for '{title}':\n" + "-" * 40)
    if isinstance(recs, str):
        print(recs)
    else:
        for i, rec in enumerate(recs, 1):
            print(f"{i:2d}. {rec}")

print_formatted_recs("Toy Story (1995)", recommend("Toy Story (1995)"))

Top Recommendations for 'Toy Story (1995)':
----------------------------------------
 1. Toy Story 2 (1999)
 2. Bug's Life, A (1998)
 3. Toy Story 3 (2010)
 4. Monsters, Inc. (2001)
 5. Finding Nemo (2003)
 6. Incredibles, The (2004)
 7. Cars (2006)
 8. Monsters University (2013)
 9. Ratatouille (2007)
10. Up (2009)


## BERT Embeddings
To challenge our TF-IDF baseline, we introduce `all-MiniLM-L6-v2`, a state-of-the-art BERT-based sentence transformer. Instead of counting explicit words, BERT processes the entire sequence to understand contextual relationships, mapping movies into a dense 384-dimensional vector space.

## BERT Similarity
Similar to our statistical model, we calculate cosine similarity. However, this time we are comparing dense semantic vectors representing the conceptual 'meaning' of the film, rather than isolated keywords.

In [50]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('/kaggle/input/datasets/inversion/sentence-transformers-222/all-MiniLM-L6-v2')

model.max_seq_length = 512
bert_embeddings = model.encode(
    movies_with_tmdb['final_features'].tolist(),
    show_progress_bar=True
)

def recommend_bert(title, top_n=10):
    if title not in indices:
        return f"{title} not found"
    
    idx = indices[title]
    
    sim_scores = cosine_similarity(
        [bert_embeddings[idx]],
        bert_embeddings
    ).flatten()
    
    sim_indices = sim_scores.argsort()[-(top_n+1):][::-1][1:]
    
    return movies_with_tmdb['title'].iloc[sim_indices]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/datasets/inversion/sentence-transformers-222/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/145 [00:00<?, ?it/s]

## BERT Recommendations
Let's observe how the deep learning model interprets the same query. You may notice subtle shifts in the generated list as the model prioritizes thematic similarity over exact tag matches.

## Balanced BERT Recommendations
Pure semantic matching can sometimes retrieve highly accurate but highly obscure results. By applying a logarithmic popularity penalty, we balance semantic relevance with general consensus, creating a more realistic product experience.

In [51]:
print_formatted_recs("Toy Story (1995) [Raw BERT]", recommend_bert("Toy Story (1995)"))

def recommend_bert_balanced(title, top_n=10, alpha=0.7):
    if title not in indices:
        return f"{title} not found"
    
    idx = indices[title]
    
    sim_scores = cosine_similarity(
        [bert_embeddings[idx]],
        bert_embeddings
    ).flatten()
    
    popularity = movies_with_tmdb['count'].values
    
    sim_scores = alpha * sim_scores + (1 - alpha) * (sim_scores / np.log1p(popularity))
    
    sim_indices = sim_scores.argsort()[-(top_n+1):][::-1][1:]
    
    return movies_with_tmdb['title'].iloc[sim_indices]

print("\n")
print_formatted_recs("Toy Story (1995) [Balanced BERT]", recommend_bert_balanced("Toy Story (1995)"))

Top Recommendations for 'Toy Story (1995) [Raw BERT]':
----------------------------------------
 1. Toy Story 2 (1999)
 2. Toy Story 3 (2010)
 3. Monsters, Inc. (2001)
 4. Up (2009)
 5. Ice Age (2002)
 6. Finding Nemo (2003)
 7. Big Hero 6 (2014)
 8. Cars (2006)
 9. Bug's Life, A (1998)
10. Simpsons Movie, The (2007)


Top Recommendations for 'Toy Story (1995) [Balanced BERT]':
----------------------------------------
 1. Toy Story 2 (1999)
 2. Toy Story 3 (2010)
 3. Monsters, Inc. (2001)
 4. Up (2009)
 5. Ice Age (2002)
 6. Finding Nemo (2003)
 7. Big Hero 6 (2014)
 8. Cars (2006)
 9. Simpsons Movie, The (2007)
10. Bug's Life, A (1998)


## User-Based Recommendations
Moving beyond item-to-item, we architect a personalized experience. We extract the embeddings of movies a specific user rated highly, mean-center those ratings to act as weights, and calculate a unified, singular 'Profile Vector'. The system then recommends movies similar to the user's overall taste profile.

In [52]:
# ================================
# 18. TRAIN TEST SPLIT
# ================================
def train_test_split(ratings, test_size=0.2):
    train_list, test_list = [], []
    
    for user_id, group in ratings.groupby('userId'):
        group = group.sample(frac=1, random_state=42)
        split_idx = int(len(group)*(1-test_size))
        
        train_list.append(group.iloc[:split_idx])
        test_list.append(group.iloc[split_idx:])
    
    return pd.concat(train_list), pd.concat(test_list)

train_ratings, test_ratings = train_test_split(ratings)

# ================================
# 19. TF-IDF USER RECOMMENDER
# ================================
def recommend_for_user(user_id, top_n=10, alpha=0.7):
    user_data = train_ratings[train_ratings['userId'] == user_id]
    
    user_data = user_data.merge(
        movies_with_tmdb[['movieId', 'final_features']],
        on='movieId'
    )
    
    user_data = user_data[user_data['rating'] >= 4.0]
    
    if len(user_data) == 0:
        return []
    
    user_tfidf = tfidf.transform(user_data['final_features'])
    weights = (user_data['rating'].values - user_data['rating'].mean()).reshape(-1,1)
    user_profile = np.asarray(user_tfidf.multiply(weights).mean(axis=0))
    sim_scores = cosine_similarity(user_profile, tfidf_matrix).flatten()
    
    popularity = movies_with_tmdb['count'].values
    sim_scores = alpha * sim_scores + (1-alpha)*(sim_scores/np.log1p(popularity))
    
    seen_movies = user_data['movieId'].values
    movie_indices = np.argsort(sim_scores)[::-1]
    
    recs = []
    for idx in movie_indices:
        movie_id = movies_with_tmdb.iloc[idx]['movieId']
        if movie_id not in seen_movies:
            recs.append(idx)
        if len(recs) >= top_n:
            break
            
    return movies_with_tmdb.iloc[recs]['title']

# ================================
# 20. BERT USER RECOMMENDER
# ================================
def build_user_profile_bert(user_id):
    user_data = train_ratings[train_ratings['userId'] == user_id]
    user_data = user_data.merge(
        movies_with_tmdb[['movieId', 'final_features']],
        on='movieId'
    )
    user_data = user_data[user_data['rating'] >= 4.0]
    
    if len(user_data) == 0:
        return None
    
    idxs = user_data['movieId'].map(
        dict(zip(movies_with_tmdb['movieId'], movies_with_tmdb.index))
    ).dropna().astype(int)
    
    user_embeddings = bert_embeddings[idxs]
    weights = (user_data['rating'].values - user_data['rating'].mean()).reshape(-1,1)
    weighted = user_embeddings * weights
    user_profile = weighted.mean(axis=0)
    
    return user_profile.reshape(1, -1)

def recommend_for_user_bert(user_id, top_n=10, alpha=0.7):
    user_profile = build_user_profile_bert(user_id)
    if user_profile is None:
        return []
    
    sim_scores = cosine_similarity(user_profile, bert_embeddings).flatten()
    
    popularity = movies_with_tmdb['count'].values
    sim_scores = alpha * sim_scores + (1 - alpha)*(sim_scores / np.log1p(popularity))
    
    seen_movies = train_ratings[train_ratings['userId'] == user_id]['movieId'].values
    movie_indices = np.argsort(sim_scores)[::-1]
    
    recs = []
    for idx in movie_indices:
        movie_id = movies_with_tmdb.iloc[idx]['movieId']
        if movie_id not in seen_movies:
            recs.append(movie_id)
        if len(recs) >= top_n:
            break
            
    # Map movie IDs back to titles for clean output
    rec_titles = movies_with_tmdb[movies_with_tmdb['movieId'].isin(recs)]['title'].tolist()
    return rec_titles

print_formatted_recs("User Profile #1 (TF-IDF)", recommend_for_user(1))

Top Recommendations for 'User Profile #1 (TF-IDF)':
----------------------------------------
 1. Cape Fear (1991)
 2. Casino (1995)
 3. Star Wars: Episode VI - Return of the Jedi (1983)
 4. Godfather, The (1972)
 5. Mean Streets (1973)
 6. Score, The (2001)
 7. Alien (1979)
 8. Analyze This (1999)
 9. Star Wars: Episode III - Revenge of the Sith (2005)
10. Deer Hunter, The (1978)


## Evaluation (Precision@10)
To quantify our results, we use **Precision@k** (where k=10). This metric measures the proportion of recommended items in the top-k set that are genuinely relevant to the user (i.e., movies they actually rated highly in the hold-out test set).

## Model Comparison
Let's put the two architectures head-to-head across our top 100 users.

In [53]:
def precision_at_k(user_id, k=10):
    recs = recommend_for_user(user_id, k)
    if len(recs) == 0:
        return None
    
    test_movies = test_ratings[
        (test_ratings['userId']==user_id) & (test_ratings['rating']>=4.0)
    ]['movieId'].values
    
    if len(test_movies)==0:
        return None
        
    rec_ids = movies_with_tmdb[
        movies_with_tmdb['title'].isin(recs)
    ]['movieId'].values
    
    return len(set(rec_ids)&set(test_movies))/k

def evaluate(num_users=100):
    users = train_ratings['userId'].value_counts()
    users = users[users>=20].index[:num_users]
    
    scores = []
    for u in users:
        p = precision_at_k(u)
        if p is not None:
            scores.append(p)
            
    return np.mean(scores)

def precision_at_k_bert(user_id, k=10):
    # Adapting evaluation to extract movie IDs directly for the metric
    user_profile = build_user_profile_bert(user_id)
    if user_profile is None: return None
    
    sim_scores = cosine_similarity(user_profile, bert_embeddings).flatten()
    popularity = movies_with_tmdb['count'].values
    sim_scores = 0.7 * sim_scores + 0.3 * (sim_scores / np.log1p(popularity))
    seen = train_ratings[train_ratings['userId'] == user_id]['movieId'].values
    idxs = np.argsort(sim_scores)[::-1]
    recs = [movies_with_tmdb.iloc[i]['movieId'] for i in idxs if movies_with_tmdb.iloc[i]['movieId'] not in seen][:k]
    
    if len(recs) == 0:
        return None
    
    test_movies = test_ratings[
        (test_ratings['userId'] == user_id) & 
        (test_ratings['rating'] >= 4.0)
    ]['movieId'].values
    
    if len(test_movies) == 0:
        return None
        
    hits = len(set(recs) & set(test_movies))
    return hits / k

def evaluate_bert(num_users=100):
    users = train_ratings['userId'].value_counts()
    users = users[users >= 20].index[:num_users]
    
    scores = []
    for u in users:
        p = precision_at_k_bert(u)
        if p is not None:
            scores.append(p)
            
    return np.mean(scores)

tfidf_score = evaluate()
bert_score = evaluate_bert()

results_data = {
    "Model Architecture": ["TF-IDF (Statistical)", "MiniLM-L6-v2 (BERT)"],
    "Embedding Type": ["Sparse (Keyword Matching)", "Dense (Semantic Meaning)"],
    "Explainability": ["High (Transparent tags)", "Low (Black Box)"],
    "Precision@10": [f"{tfidf_score:.4f}", f"{bert_score:.4f}"]
}

results_df = pd.DataFrame(results_data)

styled_results = results_df.style.set_properties(**{'text-align': 'left'}).hide(axis="index")
display(styled_results)

Model Architecture,Embedding Type,Explainability,Precision@10
TF-IDF (Statistical),Sparse (Keyword Matching),High (Transparent tags),0.2510
MiniLM-L6-v2 (BERT),Dense (Semantic Meaning),Low (Black Box),0.2900


## Insights
* **Semantic Understanding:** The BERT transformer yields higher precision. By analyzing text dynamically, it connects films that share a mood or narrative structure even if they use different vocabulary in their plot summaries.
* **The Case for TF-IDF:** While slightly lower in accuracy here, TF-IDF remains incredibly valuable. It is much less computationally expensive, faster to query in real-time, and allows engineers to easily debug *why* a movie was recommended (by analyzing exact keyword weight vectors).
* **Impact of Personalization:** By constructing a mathematical profile vector from historical 4.0+ star ratings, the system smoothly transitions from a generic database lookup to a personalized engine.

## Stats Export
To integrate this engine into a live Flask frontend, we must decouple our data science pipeline from our software architecture. We export essential matrix shapes and backend statistics as a clean JSON artifact for the API to ingest on startup.

## Conclusion
By fusing standard statistical methods with advanced deep learning embeddings, we can design scalable recommendation systems. Incorporating popularity penalization and personalized vectors ensures that recommendations remain both conceptually accurate and practically useful to end-users.

In [55]:
os.makedirs("backend/data", exist_ok=True)

stats = {
    "num_movies": len(movies),
    "num_embeddings": bert_embeddings.shape[0],
    "embedding_dim": bert_embeddings.shape[1],
    "tfidf_features": tfidf_matrix.shape[1],
}

file_path = "backend/data/stats.json"

with open(file_path, "w") as f:
    json.dump(stats, f, indent=4)

print(f"✓ Saved API artifacts seamlessly at: {os.path.abspath(file_path)}")

✓ Saved API artifacts seamlessly at: /kaggle/working/backend/data/stats.json
